<a href="https://colab.research.google.com/github/Raul-DAW/Google-Share-Optativa/blob/main/pr%C3%A1ctica_4_teachable_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Importamos la biblioteca para utilizar nuestro modelo ya que sino no funcionará el código

In [ ]:
import tf_keras
print("Keras compatible disponible")

Keras compatible disponible


# **Teachable Machine**

En esta práctica se debe exportar el modelo que se genera en Teachable Machine.

A la hora de exportar el modelo, se puede exportar para Tensorflow.

También viene un fragmento de código para usar el modelo:

```
from keras.models import load_model  # TensorFlow is required for Keras to work
from PIL import Image, ImageOps  # Install pillow instead of PIL
import numpy as np

# Disable scientific notation for clarity
np.set_printoptions(suppress=True)

# Load the model
model = load_model("keras_Model.h5", compile=False)

# Load the labels
class_names = open("labels.txt", "r").readlines()

# Create the array of the right shape to feed into the keras model
# The 'length' or number of images you can put into the array is
# determined by the first position in the shape tuple, in this case 1
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

# Replace this with the path to your image
image = Image.open("<IMAGE_PATH>").convert("RGB")

# resizing the image to be at least 224x224 and then cropping from the center
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)

# turn the image into a numpy array
image_array = np.asarray(image)

# Normalize the image
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

# Load the image into the array
data[0] = normalized_image_array

# Predicts the model
prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

# Print prediction and confidence score
print("Class:", class_name[2:], end="")
print("Confidence Score:", confidence_score)

```

**Tarea**: subir 3 imágenes propias (una clara, una ambigua, una que no pertenezca a ninguna categoría) y anotar qué predice y con qué confianza.

**¡Ojo! Hay que cargar una  versión anterior de Keras**

In [ ]:
# forzar TensorFlow a usar el comportamiento clásico de Keras 2
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
import tf_keras as keras

print("Versión de TensorFlow:", tf.__version__)
print("Versión de TF-Keras (Legacy):", keras.__version__)

try:
    # Cargamos el modelo utilizando el módulo de compatibilidad oficial tf_keras
    model = keras.models.load_model('keras_model.h5', compile=False)
    print("\n¡MODELO CARGADO CON ÉXITO!")
    model.summary()
except Exception as e:
    print(f"\nError al cargar: {e}")

Versión de TensorFlow: 2.21.0
Versión de TF-Keras (Legacy): 2.21.0

¡MODELO CARGADO CON ÉXITO!
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 3)                 128400    
                                                                 
Total params: 538608 (2.05 MB)
Trainable params: 524528 (2.00 MB)
Non-trainable params: 14080 (55.00 KB)
_________________________________________________________________


In [ ]:
import tensorflow as tf
from keras.models import load_model
from PIL import Image, ImageOps
import numpy as np

class_names = open("labels.txt", "r").readlines()
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
image = Image.open("libreta.jpg").convert("RGB")
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)
image_array = np.asarray(image)
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
data[0] = normalized_image_array

prediction = model.predict(data)
index = np.argmax(prediction)
class_name = class_names[index]
confidence_score = prediction[0][index]

# Print prediction and confidence score
print("Class:", class_name[2:], end="")
print("Confidence Score:", confidence_score)

1/1 [==============================] - 1s 877ms/step
Class: Libreta
Confidence Score: 0.9999734


**¿Cómo se usa el modelo exportado? ¿Cómo se comporta?**

---



El modelo exportado desde Teachable Machine se carga en formato Keras (`keras_model.h5`) y requiere que las imágenes de entrada se adapten exactamente a su formato de entrenamiento: reescalándolas a 224x224 píxeles, recortando si es necesario y normalizando los valores de los píxeles al rango de [-1, 1]. A partir de este array numérico, el modelo ejecuta la inferencia calculando las probabilidades para cada clase aprendida mediante una función de activación final tipo softmax.

Respecto a su comportamiento, el modelo clasifica con gran certeza (cercana al 100%) cuando la imagen contiene un objeto muy definido y similar a los de su conjunto de entrenamiento. Sin embargo, al enfrentarse a imágenes de objetos que nunca vio o a elementos ambiguos, carece de un mecanismo de "descarte" o categoría de "desconocido"; está forzado a repartir la probabilidad entre las clases que conoce, lo que puede llevarle a dar predicciones erróneas con niveles de confianza engañosamente elevados.

In [ ]:
import tf_keras as keras
from PIL import Image, ImageOps
import numpy as np

# Cargar el modelo
model = keras.models.load_model("keras_model.h5", compile=False)

# Cargar las categorías
class_names = open("labels.txt", "r", encoding="utf-8").readlines()

print("Modelo cargado correctamente")
print(class_names)

Modelo cargado correctamente
['0 Torre de Eifel\n', '1 Libreta\n', '2 Abanico\n']


In [ ]:
# Función para clasificar imágenes

def analizar_imagen(nombre_imagen):

    # Abrimos la imagen
    image = Image.open(nombre_imagen).convert("RGB")

    # Cambiamos el tamaño de la imagen
    image = ImageOps.fit(image, (224, 224), Image.Resampling.LANCZOS)

    # Convertimos la imagen en números
    image_array = np.asarray(image)

    # Preparamos la imagen para la IA
    normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1

    data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
    data[0] = normalized_image_array

    # La IA analiza la imagen
    prediction = model.predict(data, verbose=0)

    # Obtenemos el resultado
    index = np.argmax(prediction)
    class_name = class_names[index].strip()
    confidence_score = prediction[0][index]


    print("Imagen:", nombre_imagen)
    print("Categoría:", class_name)
    print("Confianza:", round(float(confidence_score) * 100, 2), "%")

    print("\nResultados de todas las categorías:")

    for i in range(len(class_names)):
        print(class_names[i].strip(), ":", round(float(prediction[0][i]) * 100, 2), "%")

In [ ]:
# implementa la clasificación con la imagen que pertenece claramente a una categoría

In [ ]:
# Primera prueba: imagen clara de una libreta

analizar_imagen("libreta.jpg")

Imagen: libreta.jpg
Categoría: 1 Libreta
Confianza: 100.0 %

Resultados de todas las categorías:
0 Torre de Eifel : 0.0 %
1 Libreta : 100.0 %
2 Abanico : 0.0 %


Para esta primera prueba utilicé una foto nítida y bien encuadrada de una libreta con el fin de evaluar la precisión del modelo en condiciones ideales.

El clasificador asignó la clase "Libreta" con un 100% de confianza (0% para el resto), verificando que cuando la imagen se ajusta fielmente a los patrones aprendidos durante el entrenamiento, la predicción es completamente acertada y sólida.

In [ ]:
# implementa la clasificación con una imagen sin ninguna categoría del modelo
# Segunda prueba: imagen desconocida

analizar_imagen("IMG_7345.jpg")

Imagen: IMG_7345.jpg
Categoría: 2 Abanico
Confianza: 90.03 %

Resultados de todas las categorías:
0 Torre de Eifel : 0.07 %
1 Libreta : 9.9 %
2 Abanico : 90.03 %


En este segundo caso introduje una imagen completamente ajena a las categorías registradas: una lata de bebida energética.

Curiosamente, el modelo clasificó la lata como "Abanico" asignándole una confianza de más del 90%. Esto pone de manifiesto una limitación habitual en este tipo de redes convolucionales: al no existir una etiqueta para "ninguna de las anteriores", el sistema intenta encajar las características visuales (bordes, contrastes, colores) en la clase más próxima dentro de su espacio de decisión, mostrando una confianza muy alta a pesar de estar equivocado.

In [ ]:
# implementa la clasificación con una imagen ambigua, que puede ser de cualquier categoría
# Tercera prueba: imagen ambigua

analizar_imagen("Ambigua.jpg")

Imagen: Ambigua.jpg
Categoría: 2 Abanico
Confianza: 52.77 %

Resultados de todas las categorías:
0 Torre de Eifel : 5.13 %
1 Libreta : 42.1 %
2 Abanico : 52.77 %


En la tercera prueba suministré una imagen ambigua o confusa para observar el reparto de probabilidades ante la falta de nitidez en los rasgos.

Los resultados reflejaron esta incertidumbre: la categoría predominante fue "Abanico" con un 52,77%, seguida muy de cerca por "Libreta" con un 42,10%. Esta distribución dividida evidencia cómo la red pierde contundencia en su predicción cuando los patrones visuales no son concluyentes, reduciéndose drásticamente el margen de seguridad entre las dos opciones más probables.